# MSci Project Test 1
We start with MGD-1K the meibography image dataset

Remember the main focus is on explainability and interpretability but there is some freedom as to where we take this project.




## Imports
We make use of multiple libraries for our project.

In [ ]:
import os
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async' # helps with out of memory issues
import numpy as np
from numpy import save, load
import matplotlib.pyplot as plt
import random
from sklearn.metrics import confusion_matrix, precision_score, recall_score, classification_report
from sklearn.utils import class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import CSVLogger
from tensorflow.keras.applications import ResNet50, ResNet50V2
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.applications import EfficientNetV2B0
from tensorflow.keras.models import load_model

import pandas as pd

import seaborn as sns

#from PIL import Image



In [ ]:
# Enable Memory Growth 
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Memory growth enabled")
    except RuntimeError as e:
        print(f"Memory growth error: {e}")

In [ ]:
!nvidia-smi
tf.config.list_physical_devices('GPU')
tf.__version__

# Data preprocessing


## Extract data with pandas 
Use Pandas to extract data (previously we tried os functions to get paths and directories)
So below we work with pandas and data frames instead, since we are already given the file names in the correct order. 

In [ ]:
# using pandas load the main large excel file with 1000 images and multiple scores and final ground truth
main_df = pd.read_excel('/home/howard/Downloads/MGD1k-main/MGD1k-main/Expore MGD1k Dataset/Graded Meiboscore/6 Round Meiboscore by MGD Experts.xlsx')
# using pandas load the csv file for the testing dataset
test_only_df = pd.read_csv('/home/howard/Downloads/MGD1k-main/MGD1k-main/Expore MGD1k Dataset/Graded Meiboscore/200 test Images used for Meiboscore Classification.txt',
                           sep = " ", header = None) # our file names have no spaces so sep=" " is okay

print("main_df ", main_df.shape)
print("test_only_df shape is ", test_only_df.shape)

In [ ]:
# For our main data frame
filenames = main_df.iloc[:, 1].values # gets all the filenames in the same order as the excel file
labels = main_df.iloc[:, 8].values # gets all the labels or ground truth values in the same order as the excel file

# We print the first 5 and double check the filename corresponds to the labels from our data(from the excel file)
print(filenames[0:5])
print(filenames.shape)
print(labels[0:5]) 


# For our test data frame
## Convert test data frame into list of filenames
test_filenames =  test_only_df.iloc[:, 0].values

print(test_filenames[0:5])
print(test_filenames.shape) # shows that is it a 2d numpy array (200 rows, 1 column)

test_list = test_filenames.tolist() # convert to list
print(test_list[0:5])
print("Length of test_list is:", len(test_list))

In [ ]:
# Filtering into training and testing datasets (dataframes)
#main_df is excel file and index 1 is the filename column)
train_df = main_df[~main_df.iloc[:, 1].isin(test_list)]
test_df = main_df[main_df.iloc[:, 1].isin(test_list)]
print(train_df)
print(test_df)

In [ ]:
print(train_df[0:7]) # see how empty values from our excel are reported as "NaN"

##  Remove NaN values
See above how empty values from our excel are reported as "NaN"

We need to filter NaN from both training and testing dataframes. Removes any row with at least one NaN.

In [ ]:
train_df_cleaned = train_df.dropna()
test_df_cleaned = test_df.dropna()

print(train_df_cleaned)
print(test_df_cleaned)

After Removing not fully gradable images from training and testing dataset we see that we get what the original researchers from MGD1K got i.e the benchmark paper see the flowchart from their website they got 752 and 189 in their datasets respectively. We successfully reproduced this.

## Creating directories and paths

In [ ]:
# For our training data frame
train_filenames = train_df_cleaned.iloc[:, 1].values # gets all the filenames in the same order as the excel file
train_labels = train_df_cleaned.iloc[:, 8].values # gets all the labels or ground truth values in the same order as the excel file

# We print the first 5 and double check the filename corresponds to the labels from our data(from the excel file)
print(train_filenames[0:5])
print(train_filenames.shape)
print(train_labels[0:5]) 


# For our testing data frame
test_filenames = test_df_cleaned.iloc[:, 1].values # gets all the filenames in the same order as the excel file
test_labels = test_df_cleaned.iloc[:, 8].values # gets all the labels or ground truth values in the same order as the excel file

# We print the first 5 and double check the filename corresponds to the labels from our data(from the excel file)
print(test_filenames[0:5])
print(test_filenames.shape)
print(test_labels[0:5]) 

We checked the ground truth labels match to the correct file name they do!

In [ ]:
# Define folder path
image_folder = '/home/howard/Downloads/MGD1k-main/MGD1k-main/Expore MGD1k Dataset/Original Images'
image_folder_eyelid_masks = '/home/howard/Downloads/MGD1k-main/MGD1k-main/Expore MGD1k Dataset/Eyelid Lebels'
image_folder_gland_masks = '/home/howard/Downloads/MGD1k-main/MGD1k-main/Expore MGD1k Dataset/Meibomian Gland Labels'

# Convert our simple list of train_filenames into directories
train_file_paths = [os.path.join(image_folder, fname) for fname in train_filenames]
train_file_paths_eyelid_masks = [os.path.join(image_folder_eyelid_masks, fname) for fname in train_filenames]
train_file_paths_gland_masks = [os.path.join(image_folder_gland_masks, fname) for fname in train_filenames]


# Convert our simply list of test_filenames into directories
test_file_paths = [os.path.join(image_folder, fname) for fname in test_filenames]###
test_file_paths_eyelid_masks = [os.path.join(image_folder_eyelid_masks, fname) for fname in test_filenames]
test_file_paths_gland_masks = [os.path.join(image_folder_gland_masks, fname) for fname in test_filenames]

print(train_file_paths[0:5], "\n", len(train_file_paths))
print(test_file_paths[0:5], "\n", len(test_file_paths))

# Note that these print in the order according to os and not our predetermined order from the excel 
print(os.listdir(image_folder)[:5])

## Fix directories using .png to .jpg
Note how the directorys are .jpg but in the excel they have labelled as .png this is a mistake we can manually correct or we can fix with python.

This is only true for the original raw images. For the eyelid masks and the meibomian gland masks they are png so it is fine.

In [ ]:
train_file_paths = [str(f).replace('.png', '.JPG') for f in train_file_paths]
test_file_paths = [str(f).replace('.png', '.JPG') for f in test_file_paths]

print(train_file_paths[0:5], "\n", len(train_file_paths))
print(test_file_paths[0:5], "\n", len(test_file_paths))

## Create our dataset/split training dataset into training and validation

In [ ]:
# increase resolution to half resolution
def load_and_preprocess(original_img_path, eyelid_mask_path, gland_mask_path, label):
    # Helper to load and decode specific formats
    def load_img(path, is_png=False):
        img = tf.io.read_file(path)
        if is_png:
            img = tf.image.decode_png(img, channels=1)
            img = tf.image.resize(img, [224, 448], method='nearest') # discrete 0 or 1 value for masks
            #img = tf.image.resize(img, [320, 640], method='nearest') # discrete 0 or 1 value for masks
        else:
            img = tf.image.decode_jpeg(img, channels=1)
            img = tf.image.resize(img, [224, 448], method='bilinear') # discrete 0 or 1 value for masks
            #img = tf.image.resize(img, [320, 640], method='bilinear') # discrete 0 or 1 value for masks
        return tf.cast(img, tf.float32) / 255.0        

    # Load each with the correct decoder
    original = load_img(original_img_path, is_png=False)
    eyelid   = load_img(eyelid_mask_path, is_png=True)
    gland    = load_img(gland_mask_path, is_png=True)

    # Apply the eyelid mask to the original image
    # This zeroes out everything outside the eyelid region for our original (image channel 1)
    masked_original = original * eyelid

    # Stack them into a 3-channel tensor (320, 640, 3)
    #combined_img = tf.concat([original, eyelid, gland], axis=-1)
    # to reset to include entire raw image set first argument to original instead of masked_original
    combined_img = tf.concat([masked_original, eyelid, gland], axis=-1) 
    
    return combined_img, label

# # halo to eyelid mask 
# def load_and_preprocess(original_img_path, eyelid_mask_path, gland_mask_path, label):
#     def load_img(path, is_png=False):
#         img = tf.io.read_file(path)
#         if is_png:
#             img = tf.image.decode_png(img, channels=1)
#             img = tf.image.resize(img, [224, 448], method='nearest')
#         else:
#             img = tf.image.decode_jpeg(img, channels=1)
#             img = tf.image.resize(img, [224, 448], method='bilinear')
#         return tf.cast(img, tf.float32) / 255.0

#     original = load_img(original_img_path, is_png=False)
#     eyelid   = load_img(eyelid_mask_path, is_png=True)
#     gland    = load_img(gland_mask_path, is_png=True)

#     # --- Create the Halo Mask ---
#     # 1. Expand dimensions to [Batch=1, Height, Width, Channel=1] for pooling
#     eyelid_4d = tf.expand_dims(eyelid, 0)
    
#     # 2. Apply Max Pooling to dilate the mask
#     # ksize and strides determine the size of the halo. 
#     # A 15x15 kernel creates a significant buffer.
#     halo_mask_4d = tf.nn.max_pool2d(
#         eyelid_4d, 
#         ksize=[1, 12, 12, 1], 
#         strides=[1, 1, 1, 1], 
#         padding='SAME'
#     )
    
#     # 3. Squeeze back to [224, 448, 1]
#     halo_mask = tf.squeeze(halo_mask_4d, axis=0)

#     # Apply the dilated halo mask instead of the tight eyelid mask
#     masked_original = original * halo_mask

#     # Stack: channel 0 is masked image, channel 1 is the original tight mask, channel 2 is gland
#     combined_img = tf.concat([masked_original, eyelid, gland], axis=-1) 
    
#     return combined_img, label

#######
## Create the dataset, we initially have a training and testing data set in 80:20 split, we split  the training set using the split ratio
## so the training set turns into a training and validation set
# 1. Define your split ratio
total_items = len(train_file_paths)
val_size = int(total_items * 0.15) 

# 2. "Glue" paths and labels together into one Dataset object
# This ensures that index 50 of paths always stays with index 50 of labels
full_dataset = tf.data.Dataset.from_tensor_slices((
    (train_file_paths, train_file_paths_eyelid_masks,train_file_paths_gland_masks),
                                                   train_labels))

# 3. Shuffle the pairs
# The buffer_size should be equal to the total number of items for a perfect shuffle
full_dataset = full_dataset.shuffle(buffer_size=total_items, seed=42, reshuffle_each_iteration=False)

# 4. Now split the zipped pairs
train_dataset = full_dataset.skip(val_size)
val_dataset = full_dataset.take(val_size)

# Since we have tuple array of paths to image we need to map those into actual images, we do this for both our train and val datasets
train_dataset = train_dataset.map(lambda paths, label: load_and_preprocess(paths[0], paths[1], paths[2], label),
                                  num_parallel_calls=tf.data.AUTOTUNE)
val_dataset = val_dataset.map(lambda paths, label: load_and_preprocess(paths[0], paths[1], paths[2], label),
                              num_parallel_calls=tf.data.AUTOTUNE)


# Use the predefined test dataset to create our test dataset, this is simple as we dont need to split the testing dataset, we leave as is
test_dataset = tf.data.Dataset.from_tensor_slices((
    (test_file_paths, test_file_paths_eyelid_masks, test_file_paths_gland_masks),
                                                   test_labels))
# As before we remap our tuple of paths into actual images
test_dataset = test_dataset.map(lambda paths, label: load_and_preprocess(paths[0], paths[1], paths[2], label),
                              num_parallel_calls=tf.data.AUTOTUNE)

#########################################################################################################

# # First, let's see what the raw dataset contains (without processing) , keep this commented out.
# for item in train_dataset.take(1):
#     print("Raw dataset structure:")
#     print("Type of item:", type(item))
#     print("Item structure:", item)
#     print("-" * 50)


for image, label in train_dataset.take(1):
    print("Image shape:", image.shape)
    print("Label shape:", label.shape)


## Data augmentation

In [ ]:
# Data augmentation
# 1. Initialize the rotation layer OUTSIDE the function
# This creates the variables only once.
rotation_layer = tf.keras.layers.RandomRotation(
    factor=0.027, # angle in degrees is x, factor = x/360
    #fill_mode='reflect'
    #fill_mode='nearest'    
    fill_mode='constant',
    fill_value=0.0
)

shift_layer = tf.keras.layers.RandomTranslation(
    height_factor=0.03, # Shift up/down by up to 3%
    width_factor=0.03,  # Shift left/right by up to 3%
    fill_mode='constant',
    fill_value=0.0
)

zoom_layer = tf.keras.layers.RandomZoom(
    height_factor=(-0.03, 0.03), # Zoom in or out by 3%
    width_factor=(-0.03, 0.03),
    fill_mode='constant',
    fill_value=0.0
)


# only applies brightness changes to first image and not masks
def augment_image(image, label):
    # 1. Split Channels: [H, W, 0] is image, [H, W, 1:3] are masks
    img_chan = image[:, :, :1]
    eyelid_mask = image[:, :, 1:2] # Channel 1 <--- Use this to "re-clean"
    mask_chans = image[:, :, 1:]

    # 2. Augment ONLY the image channel
    img_chan = tf.image.random_brightness(img_chan, max_delta=0.1)
    img_chan = tf.image.random_contrast(img_chan, lower=0.9, upper=1.1)

    # We generate noise with the same shape as the image channel
    NOISE_STDDEV = 0.02
    noise = tf.random.normal(shape=tf.shape(img_chan), mean=0.0, stddev=NOISE_STDDEV, dtype=tf.float32)
    img_chan = img_chan + noise
    
    
    # CRITICAL FIX: Clip the values back to [0, 1]
    img_chan = tf.clip_by_value(img_chan, clip_value_min=0.0, clip_value_max=1.0)

    # Get rid of positive noise, positive values outside the ROI, we apply mask to zero out that region
    img_chan = img_chan * eyelid_mask
    
    # 3. Recombine
    image = tf.concat([img_chan, mask_chans], axis=-1)

    # 4. Apply Spatial Augmentation to the WHOLE stack (Flip/Rotate)
    image = tf.image.random_flip_left_right(image)
    image = tf.expand_dims(image, 0)
    image = rotation_layer(image)
    image = shift_layer(image)  # Adds the small shifts
    image = zoom_layer(image)   # Adds the zoom in/out

    # Remove batch dimension
    image = tf.squeeze(image, 0)

    return image, label

# # halo to eyelid mask
# def augment_image(image, label):
#     # 1. Split Channels: [H, W, 0] is image, [H, W, 1:3] are masks
#     img_chan = image[:, :, :1]
#     eyelid_mask = image[:, :, 1:2] # Channel 1 <--- Use this to "re-clean"
#     mask_chans = image[:, :, 1:]

#     # 2. Augment ONLY the image channel
#     img_chan = tf.image.random_brightness(img_chan, max_delta=0.1)
#     img_chan = tf.image.random_contrast(img_chan, lower=0.9, upper=1.1)

#     # We generate noise with the same shape as the image channel
#     NOISE_STDDEV = 0.02
#     noise = tf.random.normal(shape=tf.shape(img_chan), mean=0.0, stddev=NOISE_STDDEV, dtype=tf.float32)
#     img_chan = img_chan + noise
    
    
#     # CRITICAL FIX: Clip the values back to [0, 1]
#     img_chan = tf.clip_by_value(img_chan, clip_value_min=0.0, clip_value_max=1.0)

#     # Get rid of positive noise, positive values outside the ROI, we apply mask to zero out that region
#     # --- Create the Halo Mask ---
#     # 1. Expand dimensions to [Batch=1, Height, Width, Channel=1] for pooling
#     eyelid_4d = tf.expand_dims(eyelid_mask, 0)

#     # 2. Apply Max Pooling to dilate the mask
#     # ksize and strides determine the size of the halo. 
#     # A 15x15 kernel creates a significant buffer.
#     halo_mask_4d = tf.nn.max_pool2d(
#     eyelid_4d, 
#     ksize=[1, 12, 12, 1], 
#     strides=[1, 1, 1, 1], 
#     padding='SAME'
#     )

#     # 3. Squeeze back to [224, 448, 1]
#     halo_mask = tf.squeeze(halo_mask_4d, axis=0)

#     # Apply the dilated halo mask instead of the tight eyelid mask
#     img_chan = img_chan * halo_mask
    
#     # 3. Recombine
#     image = tf.concat([img_chan, mask_chans], axis=-1)

#     # 4. Apply Spatial Augmentation to the WHOLE stack (Flip/Rotate)
#     image = tf.image.random_flip_left_right(image)
#     image = tf.expand_dims(image, 0)
#     image = rotation_layer(image)
#     image = shift_layer(image)  # Adds the small shifts
#     image = zoom_layer(image)   # Adds the zoom in/out

#     # Remove batch dimension
#     image = tf.squeeze(image, 0)

#     return image, label    

In [ ]:
# Data augmentation
# 1. Initialize the rotation layer OUTSIDE the function
# This creates the variables only once.
rotation_layer = tf.keras.layers.RandomRotation(
    factor=0.027, # angle in degrees is x, factor = x/360
    #fill_mode='reflect'
    #fill_mode='nearest'    
    fill_mode='constant',
    fill_value=0.0
)

shift_layer = tf.keras.layers.RandomTranslation(
    height_factor=0.04, # Shift up/down by up to 3%
    width_factor=0.04,  # Shift left/right by up to 3%
    fill_mode='constant',
    fill_value=0.0
)

zoom_layer = tf.keras.layers.RandomZoom(
    height_factor=(-0.03, 0.03), # Zoom in or out by 3%
    width_factor= None , # defaults to same as height if set to None, this is good otherwise its a random number so zoom in width could be different to zoom in height
    fill_mode='constant',
    fill_value=0.0
)

# halo to eyelid mask after rotation as halo before rotation means the edges at the halo when rotated generate bad info when interpolated
def augment_image(image, label):
    # 1. Intensity Augmentation (Brightness/Contrast/Noise)
    # We do this while the image is still 3D
    img_chan = image[:, :, :1]
    mask_chans = image[:, :, 1:]

    img_chan = tf.image.random_brightness(img_chan, max_delta=0.1)
    img_chan = tf.image.random_contrast(img_chan, lower=0.9, upper=1.1)

    noise = tf.random.normal(shape=tf.shape(img_chan), mean=0.0, stddev=0.02, dtype=tf.float32)
    img_chan = tf.clip_by_value(img_chan + noise, 0.0, 1.0) # CRITICAL FIX: Clip the values back to [0, 1] (VERY IMPORTANT FOR CLIPPING INSIDE THE MASK)
    
    # Recombine to apply spatial transforms to the whole stack
    image = tf.concat([img_chan, mask_chans], axis=-1)
    image = tf.image.random_flip_left_right(image)

    # 2. Spatial Augmentation (Rotate/Shift/Zoom)
    # These layers require a 4D tensor [Batch, H, W, C]
    image = tf.expand_dims(image, 0)
    image = rotation_layer(image)
    image = shift_layer(image)
    image = zoom_layer(image)

    # 3. Squeeze back to 3D for processing
    image_3d = tf.squeeze(image, axis=0)

    # 4. Slice the Rotated Channels
    img_rotated = image_3d[:, :, :1]
    eyelid_rotated = image_3d[:, :, 1:2]
    other_masks = image_3d[:, :, 2:]

    # 5. Create the "Cookie Cutter" Halo Mask from rotated eyelid
    # Reshape ensures we are strictly 4D for the MaxPool operation
    eyelid_4d_for_pool = tf.reshape(eyelid_rotated, [1, 224, 448, 1])
    
    halo_4d = tf.nn.max_pool2d(
        eyelid_4d_for_pool, 
        ksize=[1, 12, 12, 1], 
        strides=[1, 1, 1, 1], 
        padding='SAME'
    )
    
    halo_mask = tf.squeeze(halo_4d, axis=0)

    # 6. Apply Halo Mask and Recombine
    img_final = img_rotated * halo_mask
    
    # This is your final 3-channel augmented image
    image_final = tf.concat([img_final, eyelid_rotated, other_masks], axis=-1)

    return image_final, label

## Upsampling minority classes

In [ ]:
# --- STEP 1: LOAD & PREPROCESS (Base) ---
# We already have train_dataset and val_dataset mapped with load_and_preprocess.
# We add .cache() HERE. This saves the 224x448 resized images.
# If your RAM is still too full, use: .cache('/home/howard/train_cache')
train_dataset_cached = train_dataset.cache()
val_dataset_cached = val_dataset.cache()

# --- STEP 2: BALANCING (Using cached data) ---
# Isolate minority classes from the CACHED (un-augmented) data
val0_ds = train_dataset_cached.filter(lambda x, y: y == 0).take(30)
val2_ds = train_dataset_cached.filter(lambda x, y: y == 2).take(40)
val3_ds = train_dataset_cached.filter(lambda x, y: y == 3).repeat(6)

# Combine them
train_dataset_balanced = train_dataset_cached.concatenate(val0_ds).concatenate(val2_ds).concatenate(val3_ds)
train_dataset_balanced = train_dataset_balanced.cache()

## Batch size and actually applying augmentations and creating our dataset

In [ ]:
batch_size=64

# --- STEP 3: AUGMENT, SHUFFLE, BATCH ---
# We apply augmentation AFTER balancing and caching.
# This ensures new random values every epoch.
train_dataset_for_model = (
    train_dataset_balanced
    .shuffle(buffer_size=1000) 
    .map(augment_image, num_parallel_calls=tf.data.AUTOTUNE) # Fresh augmentation
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

val_dataset_for_model = (
    val_dataset_cached
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

# Test doesn't need cache unless you run it multiple times, 
# but it doesn't hurt if you have RAM.
test_dataset_for_model = test_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

Image is as we expect for 1 sample image we have a image of certain width and height
label is a scalar hence empty

In [ ]:
## helps to remove missing files, we previously created a model and trained the model which led to errors due to the missing files so we leave this here
## in memory of the missing files
for name, separate_paths in [("train_file_paths", train_file_paths), 
                             ("test_file_paths", test_file_paths),
                             ("train_file_paths_eyelid_masks", train_file_paths_eyelid_masks), 
                             ("test_file_paths_eyelid_masks", test_file_paths_eyelid_masks),
                             ("train_file_paths_gland_masks", train_file_paths_gland_masks),
                             ("test_file_paths_gland_masks", test_file_paths_gland_masks)]:
    missing_files = []
    for path in separate_paths:
        if not os.path.exists(path):
            missing_files.append(path)
            
    print(f"{name}")
    print(f"Total files in Excel: {len(separate_paths)}")
    print(f"Total missing from folder: {len(missing_files)}")
    
    if missing_files:
        print("\nFirst 3 missing files for you to check manually:")
        for f in missing_files[:3]:
            print(f" - {os.path.basename(f)}")
    
    print() # print a blank linkfor clarity


I searched for this in file explorer in the original images folder, it was labelled with .jpg instead of .JPG unlike the rest, since it was just one I fixed it manualy by renaming. Previously this code found 1 missing file for the dataset in training and no missing files in testing dataset.

## Quick checks of our dataset and check class imbalance

Random is set to 42 when shuffing the "full_dataset" and splitting into training and validation dataset. So if we restart the kernel we will get the same number occurences for each value for each dataset.

We also print the occurences of each class or value in each dataset to see class imbalance.


In [ ]:
# Extract training labels as they are in a tf.data.Dataset objects when we create the datasets and shuffle.
# We map to 'label' (the second element, index 1) and then convert to numpy
train_labels_array = np.array(list(train_dataset.map(lambda path, label: label).as_numpy_iterator())) # numpy array of training labels

# Extract validation labels
val_labels_array = np.array(list(val_dataset.map(lambda path, label: label).as_numpy_iterator())) # numpy array of validation labels

print(f"Training labels shape: {train_labels_array.shape}")
print(f"Validation labels shape: {val_labels_array.shape}")
print(f"Test labels shape: {test_labels.shape}")
####

counts = np.bincount(train_labels_array)
print("For training set:")
for i in range(4):
    print(f"Value {i}: {counts[i]} occurrences")

counts = np.bincount(val_labels_array)
print("For validation set:")
for i in range(4):
    print(f"Value {i}: {counts[i]} occurrences")

# test dataset, count occurence of each label (meiboscore)
counts = np.bincount(test_labels)
print("For testing set:")
for i in range(4):
    print(f"Value {i}: {counts[i]} occurrences")

In [ ]:
# Use the BALANCED dataset here, not the original train_dataset
train_labels_array = np.array(list(train_dataset_balanced.map(lambda img, label: label).as_numpy_iterator()))

print(f"New Training labels shape: {train_labels_array.shape}")

counts = np.bincount(train_labels_array)
print("For BALANCED training set:")
for i in range(len(counts)):
    print(f"Value {i}: {counts[i]} occurrences")

## Get or set the class weights
We now train with inverse proportional class weights, note the weights do not need to be normalised and too small weights can cause the model to learn more slowly.


In [ ]:
## Get the class weights

# 1. Get your labels as a flat array
# train_labels_array is a list or numpy array of 0, 1, 2, 3 values
weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels_array),
    y=train_labels_array     #train_labels is defautl train_labels_array is for balanced dataset
) 
# The above leads to weights=  [1.2966, 0.4187, 1.4462, 6.7143]


# # 1. Alternative: Set weights manually.
# weights = np.array([1.2966,0.4187,1.4462,4.0000])

# standard class weights when using balanced training dataset
# weights = np.array([1.3864,0.5589,1.4233, 1.2708]) 
# weights = np.array([1.3864,0.5589,1.4233, 1.7000]) # slightly higher weight for class 3

# 2. Convert to a dictionary (which Keras expects)
class_weights_dict = dict(enumerate(weights))

print("Class Weights:")
for cls, weight in class_weights_dict.items():
    print(f"  Class {cls}: {weight:.4f}")

## Create txt file (overwrites it already present)
Make sure to make a copy before running this as it deletes the old data/logs.

In [ ]:
# Save class weights first
with open("Details.txt", "w") as f:
    f.write("Class Weights:\n")
    for cls, weight in class_weights_dict.items():
        f.write(f"  Class {cls}: {weight:.4f}\n")

# Model Creation

## Model 1

In [ ]:
# # Match your exact image dimensions
# # (Height, Width, Channels)
# input_shape = (224, 448, 3)

# model = models.Sequential([
#     # First Convolutional Block
#     layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
#     layers.MaxPooling2D((2, 2)),

#     # Second Convolutional Block
#     layers.Conv2D(64, (3, 3), activation='relu'),
#     layers.MaxPooling2D((2, 2)),

#     # Third Convolutional Block
#     layers.Conv2D(128, (3, 3), activation='relu'),
#     layers.MaxPooling2D((2, 2)),

#     # Flatten and Dense layers
#     layers.Flatten(),
#     layers.Dense(128, activation='relu'),
#     layers.Dropout(0.5), # Helps prevent overfitting

#     # Output layer: 4 neurons for classes 0, 1, 2, 3
#     layers.Dense(4, activation='softmax')  # original
#     #layers.Dense(3)  # coral method with changed labels from scalar to vector 
#     #layers.Dense(1, activation='sigmoid') # use when we use the meanabsolute error as loss function
# ])

# model.summary()

## Model 2
padding="same" vs pading="valid" 
valid removes egdes, each convolution removes the edges.
same preserves the edges. Since meibomian glands can extend to the edges we try preserving this information to see if it helps.

We also add batch normalisation to help values from exploding.

In [ ]:
# input_shape = (224, 448, 3)

# model = models.Sequential([
#     # --- Block 1 ---
#     layers.Conv2D(32, (3, 3), padding='valid', input_shape=input_shape),
#     layers.BatchNormalization(),
#     layers.Activation('relu'),
#     layers.MaxPooling2D((2, 2)),

#     # --- Block 2 ---
#     layers.Conv2D(64, (3, 3), padding='valid'),
#     layers.BatchNormalization(),
#     layers.Activation('relu'),
#     layers.MaxPooling2D((2, 2)),

#     # --- Block 3 ---
#     layers.Conv2D(128, (3, 3), padding='valid'),
#     layers.BatchNormalization(),
#     layers.Activation('relu'),
#     layers.MaxPooling2D(),

#     # --- Block 4 ---
#     layers.Conv2D(128, (3, 3), padding='valid'),
#     layers.BatchNormalization(),
#     layers.Activation('relu'),
#     #layers.MaxPooling2D(),
    
#     # --- The Global Average Pooling Layer ---
#     # This replaces layers.Flatten()
#     #layers.GlobalAveragePooling2D(),
#     layers.GlobalMaxPooling2D(),
#     #layers.MaxPooling2D(),

#     # --- Fully Connected ---
#     #layers.Flatten(),
#     layers.Dense(256, activation='relu'),
#     #layers.Dense(128, activation='relu'),
#     layers.Dropout(0.1), # previously 0.5
    
#     # Output layer for 4 classes (0, 1, 2, 3)
#     layers.Dense(4, activation='softmax')
# ])

# model.summary()

## Model 3

In [ ]:
# input_shape = (224, 448, 3)

# model = models.Sequential([
#     # --- Block 1 ---
#     layers.Conv2D(32, (3, 3), padding='valid', input_shape=input_shape),
#     layers.BatchNormalization(),
#     layers.Activation('relu'),
#     layers.MaxPooling2D((2, 2)),

#     # --- Block 2 ---
#     layers.Conv2D(64, (3, 3), padding='valid'),
#     layers.BatchNormalization(),
#     layers.Activation('relu'),
#     layers.MaxPooling2D((2, 2)),

#     # --- Block 3 ---
#     layers.Conv2D(128, (3, 3), padding='valid'),
#     layers.BatchNormalization(),
#     layers.Activation('relu'),
#     layers.MaxPooling2D(),

#     # --- Block 4 ---
#     layers.Conv2D(128, (3, 3), padding='valid'),
#     layers.BatchNormalization(),
#     layers.Activation('relu'),
#     #layers.MaxPooling2D(),
    
#     # --- The Global Average Pooling Layer ---
#     # This replaces layers.Flatten()
#     #layers.GlobalAveragePooling2D(),
#     #layers.GlobalMaxPooling2D(),
#     #layers.MaxPooling2D(),
#     layers.GlobalAveragePooling2D(),

#     # --- Fully Connected ---
#     layers.Flatten(),
#     layers.Dense(256, activation='relu'),
#     #layers.Dense(128, activation='relu'),
#     layers.Dropout(0.1), # previously 0.5
    
#     # Output layer for 4 classes (0, 1, 2, 3)
#     layers.Dense(4, activation='softmax')
# ])

# model.summary()

## Model 4

In [ ]:
# input_shape = (224, 448, 3)

# model = models.Sequential([
#     # --- Block 1 ---
#     layers.Conv2D(32, (3, 3), padding='valid', input_shape=input_shape),
#     layers.BatchNormalization(),
#     layers.Activation('relu'),
#     layers.MaxPooling2D((2, 2)),

#     # --- Block 2 ---
#     layers.Conv2D(64, (3, 3), padding='valid'),
#     layers.BatchNormalization(),
#     layers.Activation('relu'),
#     layers.MaxPooling2D((2, 2)),

#     # --- Block 3 ---
#     layers.Conv2D(128, (3, 3), padding='valid'),
#     layers.BatchNormalization(),
#     layers.Activation('relu'),
#     layers.MaxPooling2D(),

#     # --- Block 4 ---
#     layers.Conv2D(128, (3, 3), padding='valid'),
#     layers.BatchNormalization(),
#     layers.Activation('relu'),
#     #layers.MaxPooling2D(),
    
#     # --- The Global Average Pooling Layer ---
#     # This replaces layers.Flatten()
#     #layers.GlobalAveragePooling2D(),
#     #layers.GlobalMaxPooling2D(),
#     #layers.MaxPooling2D(),
#     layers.GlobalAveragePooling2D(),

#     # --- Fully Connected ---
#     #layers.Flatten(),
#     layers.Dense(256, activation='relu'),
#     layers.Dropout(0.1), # previously 0.5
#     # Output layer for 4 classes (0, 1, 2, 3)
#     layers.Dense(4, activation='softmax')
# ])

# model.summary()

## Model 5 (ResNet50)

In [ ]:
# 1. Load the ResNet50 base model
# include_top=False removes the final 1000-class ImageNet layer
# weights='imagenet' uses pre-trained weights to start with
# base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 448, 3))
# base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(320, 640, 3))
# base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))


base_model = ResNet50(weights= None, include_top=False, input_shape=(224, 448, 3))
#base_model = ResNet50(weights= None, include_top=False, input_shape=(320, 640, 3))


# 2. Build the base model using ResNet50V2
#base_model = ResNet50V2(weights=None, include_top=False, input_shape=(224, 448, 3))


# # 2. Inject L2 (Do this BEFORE compiling)
# l2_strength = 0.0001
# for layer in base_model.layers:
#     if hasattr(layer, 'kernel_regularizer'):
#         layer.kernel_regularizer = tf.keras.regularizers.l2(l2_strength)


# 2. Freeze the base model (Optional but recommended for the first few epochs)
# This prevents the pre-trained weights from being destroyed early on
#base_model.trainable = False 
base_model.trainable = True


#####################################################
# # 1. Unfreeze the very beginning to adapt to Image+Mask input
# for layer in base_model.layers:
#     if layer.name == 'conv1_conv':
#         layer.trainable = True

# #Unfreeze last 10 layers
# # 2. Freeze all layers except the last N layers
# # Let's say you want to unfreeze the last 10 layers
# fine_tune_at = -10 

# for layer in base_model.layers[fine_tune_at:]:   # everything is frozen except last few layers
#     layer.trainable = True

# # 3. Verify which layers are trainable
# for layer in base_model.layers:
#     print(f"{layer.name}: {layer.trainable}")
#####################################################

# IMPORTANT: Keep BatchNormalization frozen to prevent mean/variance collapse
# unless you have a very large batch size (>32)
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False


# 3. Build your custom head
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(), # Reduces the spatial dimensions
    layers.BatchNormalization(),      # Helps with training stability
    layers.Dropout(0.5),             # CRITICAL for your overfitting issue    # 0.5 originally drop to 0.2
    layers.Dense(256, activation='relu'),
    #layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.00001)),
    layers.Dropout(0.3),                                                             # 0.3 drop to 0.1
    layers.Dense(4, activation='softmax') # Your 4 grading classes
])

## Model 5 ver 2
Moved batch normalisation below dense layer

In [ ]:
# # 1. Load the ResNet50 base model
# # include_top=False removes the final 1000-class ImageNet layer
# # weights='imagenet' uses pre-trained weights to start with
# base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 448, 3))

# # 2. Freeze the base model (Optional but recommended for the first few epochs)
# # This prevents the pre-trained weights from being destroyed early on
# base_model.trainable = False 

# # 3. Build your custom head
# model = models.Sequential([
#     base_model,
#     layers.GlobalAveragePooling2D(), # Reduces the spatial dimensions
#     layers.Dropout(0.2),             # CRITICAL for your overfitting issue    # 0.5 originally drop to 0.2
#     layers.Dense(256, activation='relu'),
#     layers.BatchNormalization(),      # Helps with training stability
#     layers.Dropout(0.1),                                                             # 0.3 drop to 0.1
#     layers.Dense(4, activation='softmax') # Your 4 grading classes
# ])

# # 4. Use a lower learning rate for transfer learning
# optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)

# model.compile(optimizer=optimizer, 
#               loss='sparse_categorical_crossentropy', 
#               metrics=['accuracy'])

## Model 5 ver 3
Make base model trainable.

In [ ]:
# # 1. Load the ResNet50 base model
# # include_top=False removes the final 1000-class ImageNet layer
# # weights='imagenet' uses pre-trained weights to start with
# base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 448, 3))

# # 2. Freeze the base model (Optional but recommended for the first few epochs)
# # This prevents the pre-trained weights from being destroyed early on
# base_model.trainable = True

# # 3. Build your custom head
# model = models.Sequential([
#     base_model,
#     layers.GlobalAveragePooling2D(), # Reduces the spatial dimensions
#     layers.Dropout(0.2),             # CRITICAL for your overfitting issue    # 0.5 originally drop to 0.2
#     layers.Dense(256, activation='relu'),
#     layers.BatchNormalization(),      # Helps with training stability
#     layers.Dropout(0.1),                                                             # 0.3 drop to 0.1
#     layers.Dense(4, activation='softmax') # Your 4 grading classes
# ])

# # 4. Use a lower learning rate for transfer learning
# optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)  #default learning rate is 0.0001

# model.compile(optimizer=optimizer, 
#               loss='sparse_categorical_crossentropy', 
#               metrics=['accuracy'])

## Model 5 Testing CBAM module

In [ ]:
# def cbam_block(cbam_feature, ratio=8):
#     """Contains the implementation of Convolutional Block Attention Module(CBAM)[cite: 146, 151]."""
    
#     # 1. Channel Attention
#     channel_axis = -1
#     channel = cbam_feature.shape[channel_axis]
    
#     shared_layer_one = layers.Dense(channel//ratio, activation='relu', kernel_initializer='he_normal', use_bias=True, bias_initializer='zeros')
#     shared_layer_two = layers.Dense(channel, kernel_initializer='he_normal', use_bias=True, bias_initializer='zeros')
    
#     avg_pool = layers.GlobalAveragePooling2D()(cbam_feature)    
#     avg_pool = layers.Reshape((1, 1, channel))(avg_pool)
#     avg_pool = shared_layer_one(avg_pool)
#     avg_pool = shared_layer_two(avg_pool)
    
#     max_pool = layers.GlobalMaxPooling2D()(cbam_feature)
#     max_pool = layers.Reshape((1, 1, channel))(max_pool)
#     max_pool = shared_layer_one(max_pool)
#     max_pool = shared_layer_two(max_pool)
    
#     cbam_feature = layers.Add()([avg_pool, max_pool])
#     cbam_feature = layers.Activation('sigmoid')(cbam_feature)
    
#     # Multiply the input feature by the channel attention map 
#     refined_feature = layers.Multiply()([cbam_feature, cbam_feature])
    
#     # 2. Spatial Attention
#     avg_pool = layers.Lambda(lambda x: tf.reduce_mean(x, axis=3, keepdims=True))(refined_feature)
#     max_pool = layers.Lambda(lambda x: tf.reduce_max(x, axis=3, keepdims=True))(refined_feature)
#     concat = layers.Concatenate(axis=3)([avg_pool, max_pool])
    
#     # Generating 2D spatial attention map [cite: 153]
#     cbam_feature = layers.Conv2D(filters=1, kernel_size=7, padding='same', activation='sigmoid', kernel_initializer='he_normal', use_bias=False)(concat)
    
#     return layers.Multiply()([refined_feature, cbam_feature])

In [ ]:
# # Load the base model
# base_model = ResNet50(weights=None, include_top=False, input_shape=(224, 448, 3))

# # Choose an intermediate layer to inject attention (shallow layers as per paper [cite: 146])
# # For ResNet50, 'conv1_relu' is a good shallow injection point
# target_layer_name = 'conv1_relu'
# intermediate_layer = base_model.get_layer(target_layer_name).output

# # Apply CBAM to the intermediate features
# attention_features = cbam_block(intermediate_layer)

# # Now, we need the rest of the ResNet to continue from this point.
# # However, Keras makes it hard to "inject" into a pre-built model. 
# # The easiest path for your Baseline is to put it at the very END of the base_model:
# x = base_model.output
# x = cbam_block(x) # Final attention refinement before the head [cite: 151]

# # 3. Build the Reconstructed Head (matching ADAM-Net deeper mapping [cite: 213, 493])
# x = layers.GlobalAveragePooling2D()(x)
# x = layers.BatchNormalization()(x)
# x = layers.Dense(2048, activation='relu')(x) # Paper uses 2048-1024-num_classes [cite: 213, 493]
# x = layers.BatchNormalization()(x)
# x = layers.Dropout(0.4)(x) # Paper uses 0.35 dropout [cite: 214]
# x = layers.Dense(1024, activation='relu')(x)
# x = layers.Dropout(0.2)(x)
# output = layers.Dense(4, activation='softmax')(x)

# model = models.Model(inputs=base_model.input, outputs=output)

## Model 6 EfficientNetV2B0

In [ ]:
# # 1. Try V2 with explicit shape
# # base_model = EfficientNetV2B0(
# #     weights='imagenet', 
# #     include_top=False, 
# #     input_shape=(224, 448, 3) 
# # )

# base_model = EfficientNetV2B0(
#     weights= None, 
#     include_top=False, 
#     input_shape=(224, 448, 3) 
# )

# base_model.trainable = True

# # IMPORTANT: Keep BatchNormalization frozen to prevent mean/variance collapse
# # unless you have a very large batch size (>32)
# for layer in base_model.layers:
#     if isinstance(layer, tf.keras.layers.BatchNormalization):
#         layer.trainable = False

# # 2. Functional API Build
# inputs = layers.Input(shape=(224, 448, 3))
# x = base_model(inputs)
# x = layers.GlobalAveragePooling2D()(x)
# x = layers.BatchNormalization()(x)
# x = layers.Dropout(0.5)(x)
# x = layers.Dense(256, activation='relu')(x)
# x = layers.Dropout(0.3)(x)
# outputs = layers.Dense(4, activation='softmax')(x)

# model = models.Model(inputs, outputs)


In [ ]:
# # 1. Try V2 with explicit shape
# # base_model = EfficientNetV2B0(
# #     weights='imagenet', 
# #     include_top=False, 
# #     input_shape=(224, 448, 3) 
# # )

# base_model = EfficientNetV2B0(
#     weights= None, 
#     include_top=False, 
#     input_shape=(224, 448, 3) 
# )

# base_model.trainable = True

# # IMPORTANT: Keep BatchNormalization frozen to prevent mean/variance collapse
# # unless you have a very large batch size (>32)
# for layer in base_model.layers:
#     if isinstance(layer, tf.keras.layers.BatchNormalization):
#         layer.trainable = False

# # 2. Functional API Build
# inputs = layers.Input(shape=(224, 448, 3))
# x = base_model(inputs)
# avg_p = layers.GlobalAveragePooling2D()(x)
# max_p = layers.GlobalMaxPooling2D()(x)
# x = layers.Concatenate()([avg_p, max_p])
# x = layers.BatchNormalization()(x)
# x = layers.Dropout(0.5)(x)
# x = layers.Dense(256, activation='relu')(x)
# x = layers.Dropout(0.3)(x)
# outputs = layers.Dense(4, activation='softmax')(x)

# model = models.Model(inputs, outputs)


In [ ]:
# # 1. Try V2 with explicit shape
# base_model = EfficientNetV2B0(
#     weights='imagenet', 
#     include_top=False, 
#     input_shape=(224, 448, 3) 
# )
# base_model.trainable = True 

# # 2. Functional API Build
# inputs = layers.Input(shape=(224, 448, 3))
# x = base_model(inputs)
# x = layers.GlobalAveragePooling2D()(x)
# # Remove the first BatchNormalization, let the base model's norm handle it
# x = layers.Dense(512, activation='relu')(x) # Increased capacity
# x = layers.BatchNormalization()(x) 
# x = layers.Dropout(0.4)(x)
# outputs = layers.Dense(4, activation='softmax')(x)

# model = models.Model(inputs, outputs)



## Model 6 Efficientnet (claude suggestions)

In [ ]:
# # ── Backbone ───────────────────────────────────────────────────────────────
# def build_efficientnet(unfreeze_from_layer=None):
#     base_model = EfficientNetV2B0(
#         weights='imagenet',       # Always start with imagenet, never None
#         include_top=False,
#         input_shape=(224, 448, 3) # Square input — fixes aspect ratio problem
#     )

#     # Re-initialise ONLY the first conv layer
#     # It needs to adapt from RGB to replicated greyscale
#     # Find the first conv layer by name
#     for layer in base_model.layers:
#         if isinstance(layer, tf.keras.layers.Conv2D):
#             first_conv = layer
#             break
#     print(f"Re-initialising: {first_conv.name}")
#     new_kernel = tf.keras.initializers.HeNormal()(shape=first_conv.kernel.shape)
#     first_conv.kernel.assign(new_kernel)
#     if first_conv.bias is not None:
#         first_conv.bias.assign(tf.zeros_like(first_conv.bias))

#     # Default: freeze everything
#     base_model.trainable = False

#     # Stage 2+: selectively unfreeze from a layer index onward
#     if unfreeze_from_layer is not None:
#         for layer in base_model.layers[unfreeze_from_layer:]:
#             layer.trainable = True

#     # Always keep first conv trainable — must adapt to greyscale
#     first_conv.trainable = True

#     # Always freeze ALL BatchNorm layers — critical for small datasets
#     # Unfreezing BN on small data causes mean/variance collapse
#     for layer in base_model.layers:
#         if isinstance(layer, tf.keras.layers.BatchNormalization):
#             layer.trainable = False

#     # ── Head ──────────────────────────────────────────────────────────────
#     inputs = layers.Input(shape=(224, 448, 3))

#     # training=False is critical — keeps BN running in inference mode
#     # even during training, preventing frozen BN from corrupting gradients
#     x = base_model(inputs, training=False)

#     # Dual pooling — avg captures distributed texture, max captures peak activations
#     # Both are useful for grading tasks
#     avg_p = layers.GlobalAveragePooling2D()(x)
#     max_p = layers.GlobalMaxPooling2D()(x)
#     x = layers.Concatenate()([avg_p, max_p])

#     x = layers.BatchNormalization()(x)  # This BN is in the head, always trainable
#     x = layers.Dropout(0.4)(x)

#     # Smaller dense layer than before — 256 was too large for your dataset size
#     x = layers.Dense(128, activation='relu',
#                      kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
#     x = layers.Dropout(0.2)(x)

#     # Softmax as requested — nominal 4-class classification
#     outputs = layers.Dense(4, activation='softmax')(x)

#     return models.Model(inputs, outputs)

# model = build_efficientnet(unfreeze_from_layer=None)

## Model 7

In [ ]:
# def gland_block(x, filters):
#     # Branch 1: Small details (3x3)
#     a = layers.Conv2D(filters, (3, 3), padding='same', activation='relu')(x)
#     # Branch 2: Wider context (5x5)
#     b = layers.Conv2D(filters, (5, 5), padding='same', activation='relu')(x)
    
#     x = layers.Concatenate()([a, b])
#     x = layers.BatchNormalization()(x)
#     return layers.MaxPooling2D((2, 2))(x)

# # Build the model
# inputs = layers.Input(shape=(224, 448, 3))

# # Initial high-res feature extraction
# x = layers.Conv2D(32, (7, 7), strides=2, padding='same', activation='relu')(inputs)

# # Blocks that preserve texture
# x = gland_block(x, 32)
# x = gland_block(x, 64)
# x = gland_block(x, 128)

# # The "Decision" Head
# x = layers.GlobalAveragePooling2D()(x) # Better for "percentage of loss"
# x = layers.Dense(256, activation='relu')(x)
# x = layers.Dropout(0.4)(x)
# outputs = layers.Dense(4, activation='softmax')(x)

# model = models.Model(inputs, outputs)

In [ ]:
# def dilated_gland_block(x, filters):
#     # Branch 1: Standard 3x3 for local texture
#     a = layers.Conv2D(filters, (3, 3), padding='same', dilation_rate=1, activation='relu')(x)
    
#     # Branch 2: Dilated 3x3 (Rate 2) to see "gaps" between glands
#     b = layers.Conv2D(filters, (3, 3), padding='same', dilation_rate=2, activation='relu')(x)
    
#     # Branch 3: Dilated 3x3 (Rate 4) to see the whole eyelid context
#     c = layers.Conv2D(filters, (3, 3), padding='same', dilation_rate=4, activation='relu')(x)
    
#     # Combine the different scales
#     combined = layers.Concatenate()([a, b, c])
#     x = layers.BatchNormalization()(combined)
    
#     # Projection layer to manage filter size
#     x = layers.Conv2D(filters, (1, 1), padding='same', activation='relu')(x)
    
#     # Use a smaller pool (or no pool) to keep resolution high
#     return layers.MaxPooling2D((2, 2))(x)


# inputs = layers.Input(shape=(224, 448, 3))

# # Initial high-res feature extraction (No pooling here!)
# x = layers.Conv2D(32, (3, 3), padding='same', activation='relu')(inputs)

# # Stack Dilated Blocks
# x = dilated_gland_block(x, 32)  # Sees small gaps
# x = dilated_gland_block(x, 64)  # Sees medium atrophy
# x = dilated_gland_block(x, 128) # Sees total gland dropout

# # Decision Head
# x = layers.GlobalAveragePooling2D()(x)
# x = layers.Dense(256, activation='relu')(x)
# x = layers.Dropout(0.5)(x)
# outputs = layers.Dense(4, activation='softmax')(x)

# model = models.Model(inputs, outputs)


## Model 8

In [ ]:
# def unet_encoder_block(x, filters):
#     # Two convs to refine the fine structures
#     x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
#     x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
#     # Save this for a skip connection if you go full U-Net
#     pool = layers.MaxPooling2D((2, 2))(x)
#     return x, pool

# inputs = layers.Input(shape=(224, 448, 3))

# # Level 1: 64 filters (Captures tiny gland edges)
# f1, p1 = unet_encoder_block(inputs, 64)

# # Level 2: 128 filters (Captures gland segments)
# f2, p2 = unet_encoder_block(p1, 128)

# # Level 3: 256 filters (Captures regional atrophy)
# f3, p3 = unet_encoder_block(p2, 256)

# # Instead of decoding, we "Fuse" these levels for Scoring
# # This is called a "Multi-scale Fusion"
# gap1 = layers.GlobalAveragePooling2D()(f1)
# gap2 = layers.GlobalAveragePooling2D()(f2)
# gap3 = layers.GlobalAveragePooling2D()(f3)

# combined = layers.Concatenate()([gap1, gap2, gap3])

# x = layers.Dense(512, activation='relu')(combined)
# x = layers.Dropout(0.5)(x)
# outputs = layers.Dense(4, activation='softmax')(x)

# model = models.Model(inputs, outputs)

## Model 9 Mobilenetv2

In [ ]:
# base_model = tf.keras.applications.MobileNetV2(
#     weights='imagenet',
#     include_top=False,
#     input_shape=(224, 448, 3)  # square crop!
# )


# # 2. Freeze the base model (Optional but recommended for the first few epochs)
# # This prevents the pre-trained weights from being destroyed early on
# #base_model.trainable = False 
# base_model.trainable = True


# #####################################################
# # # 1. Unfreeze the very beginning to adapt to Image+Mask input
# # for layer in base_model.layers:
# #     if layer.name == 'conv1_conv':
# #         layer.trainable = True

# # #Unfreeze last 10 layers
# # # 2. Freeze all layers except the last N layers
# # # Let's say you want to unfreeze the last 10 layers
# # fine_tune_at = -10 

# # for layer in base_model.layers[fine_tune_at:]:   # everything is frozen except last few layers
# #     layer.trainable = True

# # # 3. Verify which layers are trainable
# # for layer in base_model.layers:
# #     print(f"{layer.name}: {layer.trainable}")
# #####################################################

# # IMPORTANT: Keep BatchNormalization frozen to prevent mean/variance collapse
# # unless you have a very large batch size (>32)
# for layer in base_model.layers:
#     if isinstance(layer, tf.keras.layers.BatchNormalization):
#         layer.trainable = False


# # 3. Build your custom head
# model = models.Sequential([
#     base_model,
#     layers.GlobalAveragePooling2D(), # Reduces the spatial dimensions
#     layers.BatchNormalization(),      # Helps with training stability
#     layers.Dropout(0.5),             # CRITICAL for your overfitting issue    # 0.5 originally drop to 0.2
#     layers.Dense(256, activation='relu'),
#     #layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.00001)),
#     layers.Dropout(0.3),                                                             # 0.3 drop to 0.1
#     layers.Dense(4, activation='softmax') # Your 4 grading classes
# ])


## Model 10 Densenet-121

In [ ]:
# # 1. Load the DenseNet121 base model
# # DenseNet-121 is deeper but has fewer parameters than ResNet50
# # input_shape=(224, 448, 3) fits your aspect ratio well
# base_model = tf.keras.applications.DenseNet121(
#     weights=None,  # Or 'imagenet' if you want to try transfer learning
#     include_top=False, 
#     input_shape=(224, 448, 3)
# )

# # 2. Set Trainability
# # For medical masks, training from scratch (trainable=True) is often better 
# # than ImageNet weights because masks don't look like "natural" objects.
# base_model.trainable = True

# # 3. Strategy for BatchNormalization
# # DenseNet relies HEAVILY on BN layers. If your batch size is small (e.g., < 16),
# # keep these frozen to use the global statistics and prevent training instability.
# for layer in base_model.layers:
#     if isinstance(layer, tf.keras.layers.BatchNormalization):
#         layer.trainable = False

# # 4. Build the custom head
# # Note: DenseNet's output is naturally "dense" in features, 
# # so we use a robust head to map to your 4 meiboscore classes.
# model = models.Sequential([
#     base_model,
#     layers.GlobalAveragePooling2D(),
#     layers.BatchNormalization(),
    
#     # First Dense Block
#     layers.Dense(512, activation='relu'), 
#     layers.BatchNormalization(),
#     layers.Dropout(0.4), # High dropout for the large feature set
    
#     # Second Dense Block
#     layers.Dense(256, activation='relu'),
#     layers.Dropout(0.2),
    
#     # 4 classes: Grade 0, 1, 2, 3
#     layers.Dense(4, activation='softmax') 
# ])

In [ ]:
# ### This is just testing extra stuff, for any saved stuff use the above code cell

# # 1. Load the DenseNet121 base model
# # DenseNet-121 is deeper but has fewer parameters than ResNet50
# # input_shape=(224, 448, 3) fits your aspect ratio well
# base_model = tf.keras.applications.DenseNet121(
#     weights=None,  # Or 'imagenet' if you want to try transfer learning
#     include_top=False, 
#     input_shape=(224, 448, 3)
# )

# # 2. Set Trainability
# # For medical masks, training from scratch (trainable=True) is often better 
# # than ImageNet weights because masks don't look like "natural" objects.
# base_model.trainable = True

# # 3. Strategy for BatchNormalization
# # DenseNet relies HEAVILY on BN layers. If your batch size is small (e.g., < 16),
# # keep these frozen to use the global statistics and prevent training instability.
# # for layer in base_model.layers:
# #     if isinstance(layer, tf.keras.layers.BatchNormalization):
# #         layer.trainable = False

# # 4. Build the custom head
# # Note: DenseNet's output is naturally "dense" in features, 
# # so we use a robust head to map to your 4 meiboscore classes.
# model = models.Sequential([
#     base_model,
#     layers.GlobalAveragePooling2D(),
#     layers.BatchNormalization(),
    
#     # First Dense Block
#     layers.Dense(1024, activation='relu'), 
#     layers.BatchNormalization(),
#     layers.Dropout(0.35), # High dropout for the large feature set
    
#     # Second Dense Block
#     layers.Dense(512, activation='relu'),
#     layers.Dropout(0.2),

#     # Second Dense Block
#     layers.Dense(256, activation='relu'),
#     layers.Dropout(0.2),
    
#     # 4 classes: Grade 0, 1, 2, 3
#     layers.Dense(4, activation='softmax') 
# ])

## Model Compile

In [ ]:
# #default learning rate 0.001 , slower1 0.0001, slower2 0.00001
#l_rate = 0.0001
l_rate = 0.00005
#l_rate = 0.000025
#l_rate = 0.00001

model.compile(
    optimizer= tf.keras.optimizers.Adam(learning_rate=l_rate), #default learning rate for ADAM is 0.001
    loss='sparse_categorical_crossentropy',
    #loss = tf.keras.losses.MeanAbsoluteError(), # must set output neurons to 1 instead of 4
    #loss = tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=['accuracy']
)



## Model training settings
Implement early stopping (to prevent overfitting) and takes the best weights from all runs as our final model, since our metric is "validation loss" it sets that as our metric to choose the best weights. 

In [ ]:
### Training using early stop

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=5,          # Stop if val_loss doesn't improve for 5 epochs
    restore_best_weights=True
)

early_stop_12 = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=12,          # Stop if val_loss doesn't improve for 12 epochs
    restore_best_weights=True
)

early_stop_30 = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=30,          # Stop if val_loss doesn't improve for 30 epochs
    restore_best_weights=True
)


### Training reducing learing rate on  plateau
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.2,       # New LR = LR * factor (e.g., 0.001 -> 0.0002)
    patience=5,        # Wait 5 epochs of no improvement before dropping LR
    min_lr=1e-6,       # Don't let the LR go below this value
    verbose=1          # Logs a message when the LR is reduced
)

# Model training 

## Training

In [ ]:
# 1. Set up the logger
# 'append=True' ensures it adds to your history instead of overwriting it
text_file_logger = CSVLogger('Details.txt', append=True, separator='\t')

# history = model.fit(
#     train_dataset_for_model,
#     validation_data=val_dataset_for_model,
#     epochs=100,
#     class_weight=class_weights_dict, # class weights, remove this if not implementing class weights
#     callbacks=[early_stop, text_file_logger] # early stops with patience set to 5
# )

# history = model.fit(
#     train_dataset_for_model,
#     validation_data=val_dataset_for_model,
#     epochs=400,   # normally 100 or 150 or 400
#     class_weight=class_weights_dict, # class weights, remove this if not implementing class weights
#     callbacks=[early_stop_12, text_file_logger] # early stops with patience set to 12
# )

history = model.fit(
    train_dataset_for_model,
    validation_data=val_dataset_for_model,
    epochs=400,   # normally 100 or 150 or 400
    class_weight=class_weights_dict, # class weights, remove this if not implementing class weights
    callbacks=[early_stop_12, text_file_logger] # early stops with patience set to 12
    #callbacks=[early_stop_30, text_file_logger] # early stops with patience set to 30
    #callbacks=[early_stop_12, text_file_logger, reduce_lr] # early stops with patience set to 12
)


## Training history

In [ ]:
def plot_history(history):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs = range(1, len(acc) + 1)

    plt.figure(figsize=(12, 5))

    # Plot Accuracy
    plt.subplot(1, 2, 1)
    plt.plot(epochs, acc, 'bo-', label='Training Acc')
    plt.plot(epochs, val_acc, 'go-', label='Validation Acc')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    #plt.xticks(epochs)
    plt.legend()

    # Plot Loss
    plt.subplot(1, 2, 2)
    plt.plot(epochs, loss, 'bo-', label='Training Loss')
    plt.plot(epochs, val_loss, 'go-', label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    #plt.xticks(epochs)
    plt.legend()

    plt.tight_layout()
    plt.show()

# Call the function
plot_history(history)

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

print("Training Accuracy:", acc)
print("\n Validation Accuracy:", val_acc)
print("\n Training loss:", loss)
print("\n Validation loss:", val_loss)

# Model Testing

## Model evaluate output

In [ ]:
test_loss, test_accuracy = model.evaluate(test_dataset_for_model)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

## Predictions

In [ ]:
from sklearn.metrics import confusion_matrix
# Get predictions
y_pred = []
y_true = []

for imgs, labels in test_dataset_for_model:
    preds = model.predict(imgs, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())

# Check the distribution of what the model is actually "thinking"
print(f"Unique predictions made by model: {np.unique(y_pred)}")
print(f"Frequency of each prediction: {np.bincount(y_pred)}")

## Confusion Matrix

True Positive (TP): Patient has the disease, and the test says "Yes."

False Positive (FP): Patient is healthy, but the test says "Yes" (a False Alarm).

True Negative (TN): Patient is healthy, and the test says "No."

False Negative (FN): Patient has the disease, but the test says "No" (a Miss).

Since we have 4 classes not just a true or false (2 class problem) our confusion matrix is larger, but we can still redefine it such that is a true or false problem, i.e true is grade 3 is predicted as grade 3, false is grade 3 is the sum of those predicted as grade 0,1,2

In [ ]:
# 1. Get all true labels and model predictions
y_true = []
y_pred = []

for imgs, labels in test_dataset_for_model:
    preds = model.predict(imgs, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())

# 2. Create the confusion matrix
cm = confusion_matrix(y_true, y_pred)
class_names = ['Grade 0', 'Grade 1', 'Grade 2', 'Grade 3']

# 3. Plot using Seaborn
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, 
            yticklabels=class_names)

plt.title('Confusion Matrix: Meibomian Gland Grading')
plt.ylabel('Actual Grade')
plt.xlabel('Predicted Grade')
plt.show()

## Precision and recall
Precision (quality) asks: "Of all the times the model predicted 'Positive,' how often was it actually right?"

$$ Precision = \frac{TP}{TP+FP}$$

High Precision Example: A "Spam" filter. You’d rather see a few spam emails in your inbox (low recall) than have one important work email accidentally sent to the trash (low precision).

Recall (quantity) (also called Sensitivity) asks: "Of all the actual 'Positive' cases that exist, how many did the model find?"

$$ Recall = \frac{TP}{TP+FN}$$

High Recall Example: Cancer screening. It is much better to tell a healthy person they might be sick and need more tests (False Positive) than to tell a sick person they are perfectly fine (False Negative).

In [ ]:
# Get arrays of scores for each class
precisions = precision_score(y_true, y_pred, average=None)
recalls = recall_score(y_true, y_pred, average=None)

for i in range(4):
    print(f"Class {i}: Precision = {precisions[i]:.4f}, Recall = {recalls[i]:.4f}")

In [ ]:
# Define your class names for better readability
target_names = ['Class 0', 'Class 1', 'Class 2', 'Class 3']

# Print the report
print(classification_report(y_true, y_pred, target_names=target_names))

## Get actual predictions from model on test dataset

In [ ]:
# List to store the final guesses
all_predictions = []
all_softmax_values = []  # store softmax probabilities

for imgs, labels in test_dataset_for_model:
    # 1. Get raw probabilities
    preds = model.predict(imgs, verbose=0)
    
    # 2. Pick the index with the highest probability (the actual score), for softmax we skip this step
    # axis=1 tells it to look horizontally across the 4 scores
    guesses = np.argmax(preds, axis=1)
    
    # 3. Add to our list
    all_predictions.extend(guesses)
    # For softmax values instead we take softmax probabilities from preds and append to our array(not its in batches so we used extend)
    all_softmax_values.extend(preds)

# Convert to a final NumPy array
final_prediction_array = np.array(all_predictions)

# Convert to a numpy array with shape (total_samples, 4)
final_softmax_array = np.array(all_softmax_values)

# Print the final result
print("Array of Model Predictions (Meiboscores):")
print(final_prediction_array)

In [ ]:
y_true = np.array(y_true).astype(int)
print(y_true)

In [ ]:
# Print the final softmax results
print("Array of Model Softmax Scores (Probabilities for each class):")
print(final_softmax_array)

In [ ]:
with open("Details.txt", "a") as f:
    # Print the final result
    f.write(f"\n \n Array of Model Predictions softmax probabilities:\n {final_softmax_array.tolist()}")


# Save details to txt file

## Save general details and results

In [ ]:
with open("Details.txt", "a") as f:
    # Print batch size and learning rate
    f.write("\n")
    f.write(f"batch_size : {batch_size}")
    f.write("\n")
    f.write(f"l_rate : {l_rate}")
    
    # Evaluate output
    f.write("\n")  # Adds a blank line to separate from previous data
    f.write("Model Evaluation Results:\n")
    f.write(f"Test Loss: {test_loss:.4f}\n")
    f.write(f"Test Accuracy: {test_accuracy:.4f}\n")

    # Unique predictions output
    f.write(f"Unique predictions made by model: {np.unique(y_pred)}\n")
    f.write(f"Frequency of each prediction: {np.bincount(y_pred)}\n")

    # Precision and recall
    for i in range(4):
        f.write(f"Class {i}: Precision = {precisions[i]:.4f}, Recall = {recalls[i]:.4f}\n")

    # Full classification_report
    f.write(classification_report(y_true, y_pred, target_names=target_names))

    # Training history arrays
    f.write(f"\n Training Accuracy: \n {acc}")
    f.write(f"\n Validation Accuracy: \n {val_acc}")
    f.write(f"\n Training loss: \n {loss}")
    f.write(f"\n Validation loss: \n {val_loss}")

    # Print the final result
    f.write(f"\n Array of Model Predictions (Meiboscores):\n {final_prediction_array.tolist()}")


## Save model details

In [ ]:
with open("Details.txt", "a") as f:
    f.write(f"""\n

# 1. Load the ResNet50 base model
# include_top=False removes the final 1000-class ImageNet layer
# weights='imagenet' uses pre-trained weights to start with
# base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 448, 3))
# base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(320, 640, 3))
# base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))


base_model = ResNet50(weights= None, include_top=False, input_shape=(224, 448, 3))
#base_model = ResNet50(weights= None, include_top=False, input_shape=(320, 640, 3))


# 2. Build the base model using ResNet50V2
#base_model = ResNet50V2(weights=None, include_top=False, input_shape=(224, 448, 3))


# # 2. Inject L2 (Do this BEFORE compiling)
# l2_strength = 0.0001
# for layer in base_model.layers:
#     if hasattr(layer, 'kernel_regularizer'):
#         layer.kernel_regularizer = tf.keras.regularizers.l2(l2_strength)


# 2. Freeze the base model (Optional but recommended for the first few epochs)
# This prevents the pre-trained weights from being destroyed early on
#base_model.trainable = False 
base_model.trainable = True


#####################################################
# # 1. Unfreeze the very beginning to adapt to Image+Mask input
# for layer in base_model.layers:
#     if layer.name == 'conv1_conv':
#         layer.trainable = True

# #Unfreeze last 10 layers
# # 2. Freeze all layers except the last N layers
# # Let's say you want to unfreeze the last 10 layers
# fine_tune_at = -10 

# for layer in base_model.layers[fine_tune_at:]:   # everything is frozen except last few layers
#     layer.trainable = True

# # 3. Verify which layers are trainable
# for layer in base_model.layers:
#     print(f"{layer.name}: {layer.trainable}")
#####################################################

# IMPORTANT: Keep BatchNormalization frozen to prevent mean/variance collapse
# unless you have a very large batch size (>32)
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False


# 3. Build your custom head
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(), # Reduces the spatial dimensions
    layers.BatchNormalization(),      # Helps with training stability
    layers.Dropout(0.5),             # CRITICAL for your overfitting issue    # 0.5 originally drop to 0.2
    layers.Dense(256, activation='relu'),
    #layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.00001)),
    layers.Dropout(0.3),                                                             # 0.3 drop to 0.1
    layers.Dense(4, activation='softmax') # Your 4 grading classes
])

       
    """)

## Save Model best weights

In [ ]:
# # This is safe because EarlyStopping already rolled the model back to the best version
# model.save_weights('best_model_weights.weights.h5')

In [ ]:
# Save entire model as only saving weights would not work sometimes
model.save('my_complete_model.keras')

## Load Model best weights

In [ ]:
# # Must create the model first! BUT NOT COMPILE  WE COMPILE AFTER LOADING

# # Load the weights
# model.load_weights('best_model_weights.weights.h5')

In [ ]:
# When loading (No need to rebuild the Sequential layers manually!)
model = load_model('my_complete_model.keras')

# Validation loading and testing 

## Load and get results

In [ ]:
## Load the model 
model = load_model('my_complete_model.keras')

# 1. Evaluate the model on the validation dataset
val_loss, val_accuracy = model.evaluate(val_dataset_for_model)

print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")

from sklearn.metrics import confusion_matrix
import numpy as np

# 2. Get predictions for the validation set
y_pred = []
y_true = []

for imgs, labels in val_dataset_for_model:
    preds = model.predict(imgs, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())

# 3. Analyze the distribution of validation predictions
print(f"Unique predictions (Val): {np.unique(y_pred)}")
print(f"Frequency of each prediction (Val): {np.bincount(y_pred)}")

y_true_val_array = np.array(y_true)
print(y_true_val_array.tolist())

## Confusion matrix

In [ ]:
# Create the confusion matrix
cm = confusion_matrix(y_true, y_pred)
class_names = ['Grade 0', 'Grade 1', 'Grade 2', 'Grade 3']

# Plot using Seaborn
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, 
            yticklabels=class_names)

plt.title('Confusion Matrix: Meibomian Gland Grading (Validation Set)')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()

## Output meiboscore array and output softmax probabilities array

In [ ]:
# List to store the final guesses for the Validation set
val_predictions = []
val_softmax_values = []  # store softmax probabilities

# Change: iterate through the validation dataset
for imgs, labels in val_dataset_for_model:
    # 1. Get raw probabilities
    preds = model.predict(imgs, verbose=0)
    
    # 2. Pick the index with the highest probability
    guesses = np.argmax(preds, axis=1)
    
    # 3. Add to our lists
    val_predictions.extend(guesses)
    val_softmax_values.extend(preds)

# Convert to final NumPy arrays
final_val_prediction_array = np.array(val_predictions)
final_val_softmax_array = np.array(val_softmax_values)

# Print the final result
print("Array of Validation Model Predictions (Meiboscores):")
print(final_val_prediction_array)

# Optional: Check shape to ensure it matches your validation sample count
print(f"Total Validation Samples Processed: {len(final_val_prediction_array)}")

In [ ]:
# Print the final softmax results
print("Array of Model Softmax Scores (Probabilities for each class):")
print(final_val_softmax_array)

## Save the arrays

In [ ]:
with open("Validation arrays meiboscores and softmax.txt", "a") as f:
    # Print the final result
    f.write(f"\n Array of validation set true meiboscores (Meiboscores):\n {y_true_val_array.tolist()}")
    f.write(f"\n Array of validation set Model Predictions (Meiboscores):\n {final_val_prediction_array.tolist()}")
    f.write(f"\n \n Array of validation set Model Predictions softmax probabilities:\n {final_val_softmax_array.tolist()}")

In [ ]:
# 2. Print a text report for precision, recall, and f1-score
print("Classification Report:")
report = (classification_report(y_true, y_pred))
print(report)

In [ ]:
with open("Validation arrays meiboscores and softmax.txt", "a") as f:
    # Add the Classification Report
    f.write("\n\nClassification Report:\n")
    f.write(report)

# Extra training

In [ ]:
# Phase 2: Unfreeze base for fine-tuning
base_model.trainable = True
# Use lower learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history2 = model.fit(
    train_dataset_for_model,
    validation_data=val_dataset_for_model,
    epochs=10,
    callbacks=[early_stop_12,  text_file_logger]
)

In [ ]:
import matplotlib.pyplot as plt

def compare_resolutions(image_path, target_height=224, target_width=448):
    # 1. Load Original (Full Resolution)
    raw_img = tf.io.read_file(image_path)
    raw_img = tf.image.decode_jpeg(raw_img, channels=1)
    
    # 2. Load Resized (Your Preprocessed version)
    # Using 'nearest' as you did to avoid bilinear blurring of mask edges
    resized_img = tf.image.resize(raw_img, [target_height, target_width], method='nearest')
    
    # Plotting
    plt.figure(figsize=(15, 10))
    
    plt.subplot(1, 2, 1)
    plt.title(f"Original Full Resolution\n{raw_img.shape[1]}x{raw_img.shape[0]}")
    plt.imshow(tf.squeeze(raw_img), cmap='gray')
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.title(f"Downsampled (Your Code)\n{target_width}x{target_height}")
    plt.imshow(tf.squeeze(resized_img), cmap='gray')
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()

# Test with your first training path
compare_resolutions(train_file_paths[0])

In [ ]:
def visualize_augmentations(dataset, num_examples=1):
    # Take one batch from the dataset
    for images, labels in dataset.take(num_examples):
        plt.figure(figsize=(15, 10))
        
        # 1. Show Original
        plt.subplot(2, 3, 1)
        plt.imshow(images.numpy().squeeze(), cmap='gray')
        plt.title(f"Original (Grade {labels.numpy()})")
        plt.axis('off')
        
        # 2. Show 5 different augmented versions of that SAME image
        for i in range(5):
            # We call the function manually
            aug_img, _ = augment_image(images, labels)
            
            plt.subplot(2, 3, i + 2)
            plt.imshow(aug_img.numpy().squeeze(), cmap='gray')
            plt.title(f"Augmented {i+1}")
            plt.axis('off')
        plt.tight_layout()
        plt.show()

# Run it on your minority class dataset to see the Grade 3 samples
visualize_augmentations(val3_ds)

In [ ]:
plt.imshow(display_img[:, :, 0], cmap='gray')

In [ ]:
# Take one batch from your final pipeline
for images, labels in train_dataset_for_model.take(2):
    plt.figure(figsize=(15, 5))
    for i in range(3): # Show first 3 images in batch
        plt.subplot(1, 3, i+1)
        # Show Channel 0 (Raw Image)
        plt.imshow(images[i][:,:,0], cmap='gray') 
        # Overlay Channel 1 (Eyelid) in Red with alpha
        plt.imshow(images[i][:,:,1], alpha=0.3, cmap='Reds')
        # Overlay Channel 2 (Gland) in Green with alpha
        plt.imshow(images[i][:,:,2], alpha=0.3, cmap='Greens')
        plt.title(f"Label: {labels[i].numpy()}")
    plt.show()

In [ ]:
# Take one batch from your final pipeline
for images, labels in train_dataset_for_model.take(2):
    plt.figure(figsize=(15, 5))
    for i in range(3): # Show first 3 images in batch
        plt.subplot(1, 3, i+1)
        # Show Channel 0 (Raw Image)
        plt.imshow(images[i]) 
        plt.title(f"Label: {labels[i].numpy()}")
    plt.show()

In [ ]:
import sys
import pandas as pd

def mem_report():
    # Get all global variables
    gl = globals()
    # Create a list of (name, size_in_mb)
    mem_usage = []
    for name, obj in gl.items():
        # Filter out built-in modules and functions
        if not name.startswith('_') and not callable(obj):
            try:
                # Convert bytes to MB
                size = sys.getsizeof(obj) / (1024**2)
                mem_usage.append((name, size))
            except:
                pass
    
    # Sort by size and display
    df_mem = pd.DataFrame(mem_usage, columns=['Variable', 'Size (MB)'])
    return df_mem.sort_values(by='Size (MB)', ascending=False).head(10)

print(mem_report())

In [ ]:
# Take one batch from your final pipeline
for images, labels in train_dataset_for_model.take(1):
    plt.figure(figsize=(15, 5))
    for i in range(3): # Show first 3 images in batch
        plt.subplot(1, 3, i+1)
        # Show Channel 0 (Raw Image)
        plt.imshow(images[i][:,:,0], cmap='gray') 
        # Overlay Channel 1 (Eyelid) in Red with alpha
        plt.imshow(images[i][:,:,1], alpha=0.3, cmap='Reds')
        # Overlay Channel 2 (Gland) in Green with alpha
        plt.imshow(images[i][:,:,2], alpha=0.3, cmap='Greens')
        plt.title(f"Label: {labels[i].numpy()}")
    plt.show()